# Real-Time Retail Feedback Intelligence

This notebook presents a Generative AI pipeline for converting customer reviews into structured business insights.

## Goal
Compare prompt engineering strategies for retail feedback analysis:
- Zero-shot prompting
- Few-shot prompting
- Chain-of-thought style prompting
- LLM-as-a-judge evaluation
- Binary recommendation prediction

> Note: This public portfolio version removes proprietary branding, local file paths, API secrets, and execution outputs.


### **Business Context**
Retail companies receive large volumes of customer reviews across products, departments, and service touchpoints. Manually reading and categorizing this feedback is time-consuming, inconsistent, and difficult to scale. This project addresses the need for a real-time feedback intelligence system that can automatically analyze customer reviews, identify the type of issue raised, determine sentiment and urgency, summarize the feedback, and generate actionable insights for business teams.

### **Objective**

The objective of this project is to build a Generative AI-powered pipeline that analyzes retail customer reviews and converts unstructured feedback into structured business insights. The solution aims to classify reviews by category, detect sentiment, summarize the main issue, identify urgency, and support product recommendation or business action decisions using prompt engineering techniques such as Zero-Shot, Few-Shot, and Chain-of-Thought prompting.
### **Dataset Used for the Notebook**
The dataset used in this notebook contains a sample of retail customer reviews. It includes customer feedback text and related product or department information. For cost control and efficient experimentation, the project focuses on a limited sample of reviews, as recommended in the assignment instructions. The dataset is used to explore customer feedback patterns, test different prompting strategies, evaluate structured outputs, and derive actionable business insights.

### **Installing and Importing Necessary Libraries**
First, let's set up the environment by installing the required Python libraries.

In [ ]:
# Install the required libraries for the project
!pip install openai gdown -q

In [ ]:
# Import the required libraries for the project

import pandas as pd
import numpy as np
import json
import time
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

### **Data Loading**
### Loading and Understanding the Data


In [ ]:
import os

os.listdir()

In [ ]:
# This cell was used during development and is not required for the portfolio version.

In [ ]:
file_path = 'data/retail_feedback_reviews.csv'

df = pd.read_csv(file_path, sep=None, engine='python')

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
# If running in Google Colab, upload the dataset to the data/ folder or mount Google Drive manually.

### **Data Overview**

In [ ]:
# Display basic information about the dataset
print("Dataset shape:", df.shape)

# Display first five rows
df.head()

##### The dataset contains customer reviews for women’s clothing products. It includes customer age, product information, review title, review text, rating, recommendation indicator, positive feedback count, and product hierarchy fields such as Division, Department, and Class. The Review.Text column is the most important input for the Generative AI model because it contains the customer feedback to be analyzed.

### **Sanity checks**

In [ ]:
# Check missing values and duplicates
missing_values = df.isnull().sum()
duplicates = df.duplicated().sum()

print("Missing values per column:")
print(missing_values)

print("\nNumber of duplicated rows:", duplicates)

###### The sanity checks show whether the dataset contains missing values or duplicated rows. This is important before using the data for Generative AI analysis because missing review texts or duplicate feedback could affect the quality and reliability of the generated insights.

### **Data Cleaning and Preprocessing**

**Think about it:** The Review Text column is the most critical feature for our Generative AI model. What should be done with rows where this text is missing?

In [ ]:
# Create a working copy of the dataset
df_clean = df.copy()

# Remove unnecessary index column if present
if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 0'])

# Remove rows where Review.Text is missing because the LLM needs text input
df_clean = df_clean.dropna(subset=['Review.Text'])

# Fill missing optional text fields
df_clean['Title'] = df_clean['Title'].fillna('')

# Remove duplicate rows if any
df_clean = df_clean.drop_duplicates()

# Reset index
df_clean = df_clean.reset_index(drop=True)

# Check cleaned dataset
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
df_clean.head()

###### Rows with missing Review.Text were removed because the Generative AI model relies on customer review text to extract category, sentiment, summary, urgency, and business recommendations. The Title column was filled with blank values because it is useful context but not mandatory. Duplicate rows were also removed to avoid repeated feedback influencing the analysis.

### **Exploratory Data Analysis**

EDA is an important part of any project involving data. It is important to investigate and understand the data better before building a model with it. A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data. A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Questions:**

1.  What is the summary statistics of the numerical data? What can you infer about the distribution of Age, Rating, and Positive Feedback Count?
    
2.  How many unique values are there in the categorical columns like Division Name, Department Name, and Class Name?
    
3.  What is the overall distribution of product Rating? Is the dataset skewed towards positive or negative reviews?
    
4.  Which Department Name receives the highest average rating, and which receives the lowest? What might this indicate?
    
5.  What are the most common words found in highly-rated reviews (4-5 stars) versus poorly-rated reviews (1-2 stars)? (Hint: Use Word Clouds). What initial hypotheses can you form about the key drivers of customer satisfaction and dissatisfaction?

Also write your observations for each questions.

In [ ]:
# Summary statistics (Age, Rating, Positive Feedback)

df_clean[['Age', 'Rating', 'Positive.Feedback.Count']].describe()

###### Observation:
The summary statistics show that the average customer age is around the mid-40s, indicating that the dataset mainly represents middle-aged customers.

The rating distribution is skewed toward higher values, with most ratings being 4 or 5, suggesting that customers generally have a positive experience with the products.

The Positive Feedback Count has a highly skewed distribution, where most reviews receive little or no feedback, while a small number of reviews receive significantly higher engagement. This indicates that only a few reviews strongly resonate with other customers.

In [ ]:
# Unique values (categorical columns)

categorical_cols = ['Division.Name', 'Department.Name', 'Class.Name']

for col in categorical_cols:
    print(f"\n{col} - Unique values:", df_clean[col].nunique())

##### Observation:
The dataset contains multiple product divisions, departments, and classes, indicating a diverse product catalog.

The presence of multiple categories allows for segmentation of customer feedback, which is useful for identifying which product types perform well and which require improvement.

In [ ]:
# Distribution of ratings

df_clean['Rating'].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt

df_clean['Rating'].value_counts().sort_index().plot(kind='bar')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Number of Reviews')
plt.show()

##### Observation:
The rating distribution is heavily skewed towards higher ratings, especially 4 and 5. This indicates that the majority of customers are satisfied with their purchases.

However, lower ratings still exist and represent critical feedback, which is valuable for identifying issues related to product quality, fit, or delivery experience.

In [ ]:
# Department with highest / lowest rating

dept_rating = df_clean.groupby('Department.Name')['Rating'].mean().sort_values(ascending=False)

dept_rating

##### Observation:
The analysis shows that some departments consistently receive higher average ratings, indicating better customer satisfaction.

On the other hand, departments with lower average ratings may be facing issues such as poor product quality, sizing inconsistencies, or unmet customer expectations.

This insight can help the business prioritize improvements in underperforming categories.

In [ ]:
# Word analysis

from wordcloud import WordCloud

# High ratings (4-5)
high_reviews = " ".join(df_clean[df_clean['Rating'] >= 4]['Review.Text'])

# Low ratings (1-2)
low_reviews = " ".join(df_clean[df_clean['Rating'] <= 2]['Review.Text'])

# Wordcloud high
wc_high = WordCloud(width=800, height=400, background_color='white').generate(high_reviews)

plt.imshow(wc_high)
plt.axis('off')
plt.title('High Rating Reviews')
plt.show()

# Wordcloud low
wc_low = WordCloud(width=800, height=400, background_color='white').generate(low_reviews)

plt.imshow(wc_low)
plt.axis('off')
plt.title('Low Rating Reviews')
plt.show()

##### Observation:
Highly-rated reviews frequently include words such as “love”, “great”, “comfortable”, and “perfect”, indicating that customer satisfaction is strongly driven by product comfort, fit, and overall aesthetic appeal.

In contrast, low-rated reviews contain terms like “size”, “small”, “fit”, “fabric”, and “look”, highlighting common issues related to incorrect sizing, material quality, and mismatch between product expectations and actual appearance.

This suggests that sizing consistency and product quality are the main drivers of dissatisfaction, while comfort and design are the key drivers of positive customer experiences. Improving size accuracy and providing clearer product descriptions could significantly enhance customer satisfaction.

## **Building the Generative AI Pipeline**

We will now build a system to analyze the reviews. This involves setting up the AI client, designing prompts, generating structured data, and evaluating the results.

#### **Setup AI Client and Data Sample**

**Questions:**

1.  How do you initialize the OpenAI client with your API key and the correct base URL?
    

#### **Note:**

For this project, we will analyze and categorize a sample of **50 customer reviews**. This number is chosen intentionally. Since the API has a **budget limit of $20**, running prompts on very large datasets can quickly exhaust your quota—especially because this exercise may involve **multiple iterations, prompt refinements, and repeated evaluations**.

To avoid unnecessary cost and ensure efficient experimentation, we recommend the following approach:

*   **Use very small samples (5–10 reviews)** during the **initial testing phase** to validate your prompt structure and logic.
    
*   **Scale up to 50 reviews** for the **final evaluation phase**, ensuring you get enough data to compare prompting techniques without draining your budget.
    
*   This strategy helps maintain cost control while still providing meaningful insights across Zero-Shot, Few-Shot, and Chain-of-Thought techniques.
    

If your API quota gets exhausted, you may temporarily switch to another free AI assistant API. However, note that external tools may also have **rate limits** or **token caps**, so you will need to build retry logic and manage throttling within your code.

In [ ]:
import os

# Store your API key as an environment variable before running:
# export OPENAI_API_KEY="your-key"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI

# Standard OpenAI client.
# If you use a course-specific gateway or another provider, configure base_url privately in your environment.
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

#### **Prompt Engineering and Evaluation**

We will test three different prompting techniques. For each, we will create a basic version (V1) and an enhanced version (V2).

**Think about it:** Why is it important to have a consistent and robust evaluation framework? How can we use an "LLM-as-Judge" to score the quality of our generated outputs objectively?

#### **Technique 1: Zero-Shot Prompting**

**Questions:**

1.  How would you design a basic Zero-Shot prompt that asks the model for Category, Sentiment, Summary, Personalized Message, and Retail Insight?
    
2.  How can you enhance this prompt with more business context (e.g., a company name, the importance of accuracy) to create a V2 prompt?
    
3.  How will you loop through the data sample to generate and store the structured output for both prompt versions?
    
4.  How will you apply the LLM-as-Judge to generate a evaluation score between 0 to 1 (decimal allowed) for the outputs and calculate the average score of V1 and V2 prompt?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Zero-Shot Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

In [ ]:
# Create a clean sample of 50 reviews for prompt evaluation

sample_df = df_clean[
    ['Review.Text', 'Rating', 'Department.Name', 'Class.Name']
].dropna(subset=['Review.Text']).sample(50, random_state=1).reset_index(drop=True)

sample_df.head()

In [ ]:
import json
import time

def call_llm(prompt, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
def zero_shot_prompt_v1(review):
    return f"""
Analyze the following retail customer review.

Review:
{review}

Return the answer in valid JSON with exactly these keys:
- Category
- Sentiment
- Summary
- Personalized_Message
- Retail_Insight
"""

In [ ]:
def zero_shot_prompt_v2(review, department, class_name):
    return f"""
You are a retail customer feedback analyst.

Business context:
The company wants to understand customer reviews quickly in order to improve products, service quality, and customer retention.

Product information:
Department: {department}
Class: {class_name}

Customer review:
{review}

Task:
Analyze the review and return only valid JSON with exactly these keys:
- Category: one of Fit, Quality, Delivery, Price, Service, Positive Feedback, Other
- Sentiment: one of Positive, Negative, Neutral
- Summary: one short sentence describing the main issue or praise
- Personalized_Message: a short professional message the retailer could send to the customer
- Retail_Insight: one practical business insight for the retail team

Do not add explanations outside the JSON.
"""

In [ ]:
results = []

for i, row in sample_df.iterrows():
    review = row['Review.Text']
    department = row['Department.Name']
    class_name = row['Class.Name']

    output_v1 = call_llm(zero_shot_prompt_v1(review))
    time.sleep(1)

    output_v2 = call_llm(zero_shot_prompt_v2(review, department, class_name))
    time.sleep(1)

    results.append({
        "review_id": i,
        "review_text": review,
        "rating": row['Rating'],
        "department": department,
        "class_name": class_name,
        "zero_shot_v1_output": output_v1,
        "zero_shot_v2_output": output_v2
    })

zero_shot_results = pd.DataFrame(results)
zero_shot_results.head()

In [ ]:
def judge_prompt(review, model_output):
    return f"""
You are evaluating the quality of an AI-generated analysis of a retail customer review.

Original review:
{review}

AI-generated output:
{model_output}

Score the output from 0 to 1 based on:
- Relevance to the review
- Correct sentiment interpretation
- Business usefulness
- Clarity
- Completeness of the requested fields

Return only a number between 0 and 1.
"""

In [ ]:
v1_scores = []
v2_scores = []

for i, row in zero_shot_results.iterrows():
    score_v1 = call_llm(judge_prompt(row['review_text'], row['zero_shot_v1_output']))
    time.sleep(1)

    score_v2 = call_llm(judge_prompt(row['review_text'], row['zero_shot_v2_output']))
    time.sleep(1)

    v1_scores.append(float(score_v1.strip()))
    v2_scores.append(float(score_v2.strip()))

zero_shot_results['v1_judge_score'] = v1_scores
zero_shot_results['v2_judge_score'] = v2_scores

zero_shot_results[['v1_judge_score', 'v2_judge_score']].mean()

##### Observation:
The evaluation results indicate that Zero-Shot Prompt V1 achieved a higher average judge score (0.978) compared to Version 2 (0.912). This suggests that the simpler and more direct prompt structure in V1 allowed the model to generate outputs that were more aligned with the evaluation criteria, particularly in terms of clarity and relevance.

In contrast, although Version 2 included additional business context and stricter instructions, it may have introduced unnecessary complexity or constraints that slightly reduced output consistency or clarity. This highlights that, in some cases, adding more context does not always improve performance and can lead to diminishing returns.

Overall, this result emphasizes the importance of balancing instruction clarity and prompt complexity when designing effective Zero-Shot prompts.

#### **Technique 2: Few-Shot Prompting**

**Questions:**

1.  How do you structure a Few-Shot prompt? What kind of examples (e.g., one positive, one negative) would be most effective?
    
2.  For the V2 prompt, how can you add a set of "rules" to guide the model's output for each field, reducing ambiguity?
    
3.  After generating and scoring the outputs, how does the performance of Few-Shot prompting compare to previous version?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your ** Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

In [ ]:
def few_shot_prompt_v1(review):
    return f"""
Analyze the retail customer review and return valid JSON.

Example 1:
Review: "The dress fits perfectly and the fabric feels very comfortable."
Output:
{{
  "Category": "Fit",
  "Sentiment": "Positive",
  "Summary": "The customer is satisfied with the fit and fabric comfort.",
  "Personalized_Message": "Thank you for your feedback. We are happy to hear that the dress fits well and feels comfortable.",
  "Retail_Insight": "Good fit and comfortable fabric are key drivers of customer satisfaction."
}}

Example 2:
Review: "The item arrived late and the quality was poor."
Output:
{{
  "Category": "Delivery",
  "Sentiment": "Negative",
  "Summary": "The customer is dissatisfied with late delivery and poor quality.",
  "Personalized_Message": "We are sorry for the late delivery and quality issue. We will review this case and work to improve your experience.",
  "Retail_Insight": "Delivery delays and quality issues can reduce customer satisfaction and should be monitored closely."
}}

Now analyze this review:
Review: {review}

Return only valid JSON with the same five keys.
"""

In [ ]:
def few_shot_prompt_v2(review, department, class_name):
    return f"""
You are a retail customer feedback analyst.

Business context:
The company wants to convert customer reviews into structured insights for product improvement, service improvement, and customer retention.

Product information:
Department: {department}
Class: {class_name}

Rules:
- Category must be one of: Fit, Quality, Delivery, Price, Service, Positive Feedback, Other.
- Sentiment must be one of: Positive, Negative, Neutral.
- Summary must be one short sentence.
- Personalized_Message must be polite, professional, and customer-facing.
- Retail_Insight must be useful for a retail business team.
- Return only valid JSON. Do not add text outside the JSON.

Example 1:
Review: "This shirt is very soft, fits well, and looks exactly like the picture."
Output:
{{
  "Category": "Fit",
  "Sentiment": "Positive",
  "Summary": "The customer is satisfied with the shirt's fit, softness, and appearance.",
  "Personalized_Message": "Thank you for your feedback. We are glad the shirt fits well and met your expectations.",
  "Retail_Insight": "Fit, comfort, and accurate product images are important drivers of positive customer experience."
}}

Example 2:
Review: "I ordered a medium but it was too small and the material felt cheap."
Output:
{{
  "Category": "Fit",
  "Sentiment": "Negative",
  "Summary": "The customer is unhappy with the sizing and material quality.",
  "Personalized_Message": "We are sorry the item did not meet your expectations. We will review the sizing and material feedback with our product team.",
  "Retail_Insight": "Sizing accuracy and material quality should be monitored to reduce returns and dissatisfaction."
}}

Now analyze this review:
Review: {review}

Return only valid JSON.
"""

In [ ]:
few_shot_results = []

for i, row in sample_df.iterrows():
    review = row['Review.Text']
    department = row['Department.Name']
    class_name = row['Class.Name']

    output_v1 = call_llm(few_shot_prompt_v1(review))
    time.sleep(1)

    output_v2 = call_llm(few_shot_prompt_v2(review, department, class_name))
    time.sleep(1)

    few_shot_results.append({
        "review_id": i,
        "review_text": review,
        "rating": row['Rating'],
        "department": department,
        "class_name": class_name,
        "few_shot_v1_output": output_v1,
        "few_shot_v2_output": output_v2
    })

few_shot_results = pd.DataFrame(few_shot_results)
few_shot_results.head()

In [ ]:
few_v1_scores = []
few_v2_scores = []

for i, row in few_shot_results.iterrows():
    score_v1 = call_llm(judge_prompt(row['review_text'], row['few_shot_v1_output']))
    time.sleep(1)

    score_v2 = call_llm(judge_prompt(row['review_text'], row['few_shot_v2_output']))
    time.sleep(1)

    few_v1_scores.append(float(score_v1.strip()))
    few_v2_scores.append(float(score_v2.strip()))

few_shot_results['few_shot_v1_judge_score'] = few_v1_scores
few_shot_results['few_shot_v2_judge_score'] = few_v2_scores

few_shot_results[['few_shot_v1_judge_score', 'few_shot_v2_judge_score']].mean()

In [ ]:
comparison_scores = pd.DataFrame({
    "Technique": ["Zero-Shot V1", "Zero-Shot V2", "Few-Shot V1", "Few-Shot V2"],
    "Average Judge Score": [
        zero_shot_results['v1_judge_score'].mean(),
        zero_shot_results['v2_judge_score'].mean(),
        few_shot_results['few_shot_v1_judge_score'].mean(),
        few_shot_results['few_shot_v2_judge_score'].mean()
    ]
})

comparison_scores

##### Observation:
The comparison results show that Zero-Shot V1 achieved the highest average score (0.978), followed closely by Few-Shot V1 (0.974) and Few-Shot V2 (0.958), while Zero-Shot V2 performed the lowest (0.912).

This indicates that adding examples through Few-Shot prompting improves consistency compared to more complex instruction-based prompts (Zero-Shot V2), but does not necessarily outperform a well-designed simple Zero-Shot prompt (V1). The strong performance of Zero-Shot V1 suggests that clarity and conciseness in instructions are critical factors for high-quality outputs.

Although Few-Shot prompting provides structured guidance through examples, the marginal performance gain over Zero-Shot V1 is limited in this case. This may indicate that the task is sufficiently straightforward for the model to generalize without needing examples, or that the examples provided did not significantly enhance the model’s reasoning.

Overall, these results highlight that prompt effectiveness depends more on clarity and alignment with the task than on the complexity or quantity of instructions, and that simpler prompts can sometimes outperform more elaborate designs.

#### **Technique 3: Chain-of-Thought (CoT) Prompting**

**Questions:**

1.  How do you instruct the model to "think step-by-step" internally but only show the final, structured answer?
    
2.  How can you combine the CoT instruction with more detailed reasoning from the COT V1 prompt to create a powerful CoT V2 prompt?
    
3.  Does encouraging the model to reason first lead to a measurable improvement in the quality of the generated insights?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

In [ ]:
def cot_prompt_v1(review):
    return f"""
You are a retail customer feedback analyst.

Analyze the following review carefully.

Think step-by-step internally about:
- What the customer is talking about
- Whether the sentiment is positive, negative, or neutral
- The key issue or praise

IMPORTANT:
Do NOT show your reasoning.

Only return the final answer in valid JSON with these keys:
- Category
- Sentiment
- Summary
- Personalized_Message
- Retail_Insight

Review:
{review}
"""

In [ ]:
def cot_prompt_v2(review, department, class_name):
    return f"""
You are an expert retail analyst.

Business goal:
Convert customer reviews into structured insights to improve products, customer experience, and business decisions.

Product context:
Department: {department}
Class: {class_name}

Instructions:
Think step-by-step internally before answering:
1. Identify the main topic (fit, quality, delivery, etc.)
2. Determine sentiment
3. Identify key issue or praise
4. Translate into business insight

IMPORTANT:
- Do NOT show your reasoning
- Only return final JSON

Rules:
- Category: Fit, Quality, Delivery, Price, Service, Positive Feedback, Other
- Sentiment: Positive, Negative, Neutral
- Summary: 1 sentence
- Personalized_Message: professional and polite
- Retail_Insight: actionable business insight

Review:
{review}

Return only valid JSON.
"""

In [ ]:
cot_results = []

for i, row in sample_df.iterrows():
    review = row['Review.Text']
    department = row['Department.Name']
    class_name = row['Class.Name']

    output_v1 = call_llm(cot_prompt_v1(review))
    time.sleep(1)

    output_v2 = call_llm(cot_prompt_v2(review, department, class_name))
    time.sleep(1)

    cot_results.append({
        "review_id": i,
        "review_text": review,
        "rating": row['Rating'],
        "department": department,
        "class_name": class_name,
        "cot_v1_output": output_v1,
        "cot_v2_output": output_v2
    })

cot_results = pd.DataFrame(cot_results)
cot_results.head()

In [ ]:
cot_v1_scores = []
cot_v2_scores = []

for i, row in cot_results.iterrows():
    score_v1 = call_llm(judge_prompt(row['review_text'], row['cot_v1_output']))
    time.sleep(1)

    score_v2 = call_llm(judge_prompt(row['review_text'], row['cot_v2_output']))
    time.sleep(1)

    cot_v1_scores.append(float(score_v1.strip()))
    cot_v2_scores.append(float(score_v2.strip()))

cot_results['cot_v1_judge_score'] = cot_v1_scores
cot_results['cot_v2_judge_score'] = cot_v2_scores

cot_results[['cot_v1_judge_score', 'cot_v2_judge_score']].mean()

In [ ]:
final_comparison = pd.DataFrame({
    "Technique": [
        "Zero-Shot V1", "Zero-Shot V2",
        "Few-Shot V1", "Few-Shot V2",
        "CoT V1", "CoT V2"
    ],
    "Average Score": [
        zero_shot_results['v1_judge_score'].mean(),
        zero_shot_results['v2_judge_score'].mean(),
        few_shot_results['few_shot_v1_judge_score'].mean(),
        few_shot_results['few_shot_v2_judge_score'].mean(),
        cot_results['cot_v1_judge_score'].mean(),
        cot_results['cot_v2_judge_score'].mean()
    ]
})

final_comparison

##### Observation:
The results show that the Zero-Shot V1 and CoT V1 approaches achieve the best performance, with an identical average score of 0.978, indicating that simple, well-structured instructions are sufficient to produce high-quality analyses in this context.

The introduction of additional complexity in the V2 versions (adding business context or detailed rules) leads to a drop in performance (e.g., Zero-Shot V2: 0.912; CoT V2: 0.932), suggesting that overly restrictive prompts can hinder the clarity or relevance of the generated responses.

Regarding Few-Shot prompting, performance remains high (0.974 for V1 and 0.958 for V2), but does not provide a significant improvement over simple Zero-Shot, indicating that the provided examples were not necessary for this relatively straightforward task.

Finally, although Chain-of-Thought encourages structured reasoning, it offers no measurable gain over Zero-Shot, confirming that this technique is more relevant for complex problems requiring multi-step reasoning.

In conclusion, the simplicity and clarity of the prompts appear to be the most decisive factors, and advanced techniques should be used in a targeted manner depending on the actual complexity of the problem.

## **Applying GenAI for Product Recommendation:**

Now, let's use the model for a different task: predicting the Recommended IND flag.

**Questions:**

1.  How do you design a prompt that strictly asks for a binary output (1 or 0) and a brief reason?
    
2.  What kind of function is needed to reliably parse the model's text response to extract the 1/0 flag and the Reason?
    
3.  How do you evaluate the model's performance as a classifier using standard metrics like accuracy, confusion matrix, and classification report?

**How the Process Works**


**1\. Prepare Data**

Copy the dataset, store the original recommendation labels, and remove them from the model input to avoid leakage.

**2\. Generate Predictions**

Use a strict two-line prompt to make the LLM output a binary recommendation (1/0) and a short reason based only on the review text.

**3\. Parse Outputs**

Extract the flag and reason from the raw LLM response using regex-based parsing that handles formatting issues.

**4\. Build Prediction Table**

Run the prompt for each review, parse the result, and store the predictions in a new DataFrame.

 **5\. Evaluate Performance**

Compare LLM predictions with true labels using accuracy, confusion matrix, and classification report.

 **6\. Explain Mismatches**

For incorrect predictions, generate a short explanation describing why the model’s decision may have differed from the human label.

In [ ]:
def build_recommendation_prompt(review):
    return f"""
You are a product recommendation assistant.

Based ONLY on the customer review below, decide if the product should be recommended.

Review:
{review}

Rules:
- Output MUST be exactly in this format:
Recommendation: 0 or 1
Reason: one short sentence

Where:
1 = Recommended
0 = Not Recommended
"""

In [ ]:
import re

def parse_recommendation_output(text):
    try:
        rec_match = re.search(r"Recommendation:\s*([01])", text)
        reason_match = re.search(r"Reason:\s*(.*)", text)

        recommendation = int(rec_match.group(1)) if rec_match else None
        reason = reason_match.group(1).strip() if reason_match else ""

        return recommendation, reason

    except:
        return None, ""

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
results_df = sample_df.copy()
results_df['y_true'] = sample_df['Recommended.IND']
results_df['y_pred'] = predictions
results_df['reason'] = reasons

In [ ]:
# Nettoyage
results_df = results_df.dropna(subset=['y_pred'])

y_true = results_df['y_true']
y_pred = results_df['y_pred']

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print("Accuracy:", accuracy)

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

##### Observation:
The model achieves an overall accuracy of 0.86, indicating strong performance in predicting product recommendations based solely on review text.

The confusion matrix shows that the model correctly identifies most positive recommendations (true positives), with a relatively high recall of 0.85 for class 1. This suggests that the model is effective at detecting when products should be recommended. However, the precision for class 0 (0.62) is lower, indicating that the model tends to misclassify some non-recommended products as recommended.

This imbalance suggests a bias toward predicting positive recommendations, likely because the dataset contains more favorable reviews or because positive sentiment is easier to detect from text.

Overall, while the model performs well as a zero-shot classifier, improvements could be made in distinguishing negative or less enthusiastic reviews, particularly in borderline or nuanced cases.

**Visualization of Sentiments Distribution**

 After generating results from all prompting techniques, it's crucial to visualize their outputs to better understand their behavior and performance. This helps us see if one technique tends to be more cautious (e.g., assigning more 'Neutral' sentiments) or if they generally agree on the sentiment of the reviews.
    
 **Questions:**
    
* How does the distribution of predicted Sentiment (Positive, Negative, Neutral) compare across the V2 versions of Zero-Shot, Few-Shot, and Chain-of-Thought? (Hint: Create a separate bar chart for each technique's V2 sentiment column).
    
* Are there noticeable differences in the counts? For example, does one technique identify more "Neutral" reviews than the others? What might this imply about its ability to handle nuance?
    


In [ ]:
print(zero_shot_results.columns)
print(few_shot_results.columns)
print(cot_results.columns)

In [ ]:
import pandas as pd
import re

def extract_sentiment(text):
    if pd.isna(text):
        return "Unknown"

    text = str(text)

    match = re.search(r"Sentiment\s*:\s*(Positive|Negative|Neutral)", text, re.IGNORECASE)

    if match:
        return match.group(1).capitalize()
    else:
        return "Unknown"

zero_shot_results['sentiment_v2'] = zero_shot_results['zero_shot_v2_output'].apply(extract_sentiment)
few_shot_results['sentiment_v2'] = few_shot_results['few_shot_v2_output'].apply(extract_sentiment)
cot_results['sentiment_v2'] = cot_results['cot_v2_output'].apply(extract_sentiment)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

sentiment_comparison = pd.DataFrame({
    "Zero-Shot V2": zero_shot_results['sentiment_v2'].value_counts(),
    "Few-Shot V2": few_shot_results['sentiment_v2'].value_counts(),
    "CoT V2": cot_results['sentiment_v2'].value_counts()
}).fillna(0)

sentiment_comparison.plot(kind="bar", figsize=(10, 6))
plt.title("Sentiment Distribution Across V2 Prompting Techniques")
plt.ylabel("Number of Reviews")
plt.xlabel("Sentiment")
plt.xticks(rotation=0)
plt.show()

sentiment_comparison

In [ ]:
zero_shot_counts = zero_shot_results['sentiment_v2'].value_counts()
few_shot_counts = few_shot_results['sentiment_v2'].value_counts()
cot_counts = cot_results['sentiment_v2'].value_counts()

In [ ]:
df.head()
df.columns

##### Observation:
### Observation:

The sentiment distribution shows how the three V2 prompting techniques classify customer reviews across Positive, Negative, and Neutral categories. Comparing the counts helps identify whether one technique is more cautious or more nuanced than the others.

If one method assigns more reviews to the Neutral class, it may indicate a stronger ability to handle mixed or ambiguous feedback. In contrast, a method that produces mostly Positive or Negative outputs may be more decisive but potentially less sensitive to nuanced customer opinions.

Overall, this visualization helps assess the consistency of the prompting techniques and supports the selection of the most reliable approach for business use.

##  **Comparison of Prompting Techniques:**
    
   *   How do the three techniques (Zero-Shot, Few-Shot, CoT) compare in terms of their responses. Use LLM to give verdict?
        
  *   Which technique was the most reliable and consistent? Why do you think it performed the best?
        
   *   What model and prompt design would you propose for a production environment?
        


##### 1. Overall Comparison (Zero-Shot vs Few-Shot vs CoT)

The three prompting techniques show different behaviors in terms of structure, consistency, and interpretability:

Zero-Shot prompting performs well in generating quick and reasonably accurate outputs, but its responses can lack consistency and sometimes miss important details due to the absence of guidance.
Few-Shot prompting improves significantly over Zero-Shot by providing examples, leading to more structured, consistent, and context-aware outputs.
Chain-of-Thought (CoT) prompting encourages deeper reasoning, which helps in handling more complex or ambiguous reviews. However, it may sometimes introduce unnecessary verbosity or slight inconsistencies if not well constrained.

##### 2. Most Reliable and Consistent Technique

Based on the evaluation scores:

Few-Shot V2 appears to be the most balanced and reliable approach.

Why it performs best:

It benefits from explicit examples, reducing ambiguity.
It produces more structured and predictable outputs.
It maintains a good balance between accuracy and consistency without overcomplicating reasoning.

Although Zero-Shot V1 scored slightly higher, it is likely:

Less stable across different inputs
More sensitive to variations in phrasing

##### 3. LLM-as-Judge Verdict

Using the LLM-as-Judge scoring framework:

All techniques perform at a relatively high level (>0.90)
However, Few-Shot and CoT approaches provide more robust outputs in terms of completeness and business usefulness
The differences in scores suggest that prompt design has a measurable but moderate impact on performance

##### 4. Recommendation for Production Environment

For a real-world retail application, the best approach would be:

Use Few-Shot Prompting (V2) with structured output format

Recommended setup:

- Technique: Few-Shot V2
- Model: A cost-efficient model (e.g., GPT-4o-mini or equivalent)
- Output format: Strict structured format (JSON or labeled fields)
- Add-ons:
   - Regex-based validation (to ensure output consistency)
   - Fallback handling for parsing errors
   - Batch processing with retry logic

Why this choice:

- High consistency and reliability
- Easier to scale and maintain
- Lower risk of unexpected outputs compared to CoT
- Better cost-performance tradeoff

### **Observations and Insights**

 **Refined Insights:**
    
   *   What are the most meaningful and recurring insights from the customer reviews, as identified by your best-performing model?

##### Refined Insights from Customer Reviews

From the best-performing model:

- Positive reviews are often driven by:
    - Fit and comfort
    - Product quality
    - Aesthetic appeal (style, color)
- Negative reviews frequently mention:
    - Size mismatch
    - Poor material quality
    - Unexpected product differences
- Neutral or mixed reviews indicate:
    - Trade-offs between style and fit
    - Inconsistent quality perception

##### Customer satisfaction is primarily influenced by fit accuracy and perceived quality, while dissatisfaction is strongly linked to expectation gaps (especially sizing and material).

##### Strategic Implication

Retailers should prioritize:

- Better size guidance systems
- Clear product descriptions
- Quality consistency

# Generating Actionable Product Improvement Suggestions


 *   Based on the aggregated insights from your best model, what are 3 short-term (3-6 months) and 3 long-term (6-12 months) actionable business recommendations for the retail company?
        
 *   How does this automated GenAI pipeline solve the initial business problem and create value?

### **Observations and Insights**:
#### Generating Actionable Product Improvement Suggestions
Short-Term Recommendations (3–6 months)
1. Improve Size Guidance and Fit Accuracy
  - Introduce clearer size charts, fit recommendations, and customer fit feedback (e.g., “runs small/large”).
  - This directly addresses one of the most frequent sources of dissatisfaction.
2. Enhance Product Descriptions and Visual Transparency
  - Provide more detailed information on fabric, quality, and actual product appearance.
  - Include real customer photos to reduce expectation gaps.
3. Implement Review-Based Feedback Loop
  - Use the GenAI pipeline to continuously monitor reviews and flag recurring issues (e.g., sizing, quality defects).
  - Enable faster response from product and quality teams.

##### Long-Term Recommendations (6–12 months)
1. Develop Data-Driven Product Design Improvements
- Use aggregated insights to adjust product design (fit, materials, durability).
- Prioritize categories with consistently negative feedback.
2. Personalized Recommendation System Enhancement
- Integrate sentiment and review insights into recommendation engines to improve product relevance and customer satisfaction.
3. Quality Control and Supplier Optimization
- Identify patterns of dissatisfaction linked to specific products or suppliers.
- Strengthen quality assurance processes across the supply chain.

##### Business Value of the GenAI Pipeline

This automated GenAI pipeline provides significant value by:

- Scalability: Enables analysis of large volumes of customer feedback in real time.
- Consistency: Standardizes review interpretation across categories and products.
- Insight Generation: Transforms unstructured text into structured, actionable insights (sentiment, issues, recommendations).
- Decision Support: Helps product, marketing, and operations teams make data-driven decisions.
- Customer Experience Improvement: Identifies key drivers of satisfaction and dissatisfaction, enabling targeted improvements.

##### Key Takeaway

The GenAI pipeline bridges the gap between raw customer feedback and strategic decision-making by converting qualitative insights into measurable business actions, ultimately improving product quality, customer satisfaction, and operational efficiency.

## **Conclusion**

This project demonstrates how Generative AI can effectively transform unstructured customer reviews into structured, actionable business insights.

By comparing Zero-Shot, Few-Shot, and Chain-of-Thought prompting techniques, we observed that prompt design has a direct impact on output quality, consistency, and business relevance. Among the tested approaches, Few-Shot prompting offered the best balance between accuracy, reliability, and scalability.

Beyond model performance, the most important outcome of this pipeline is its ability to operationalize customer feedback. It enables organizations to systematically identify key drivers of satisfaction and dissatisfaction, reduce ambiguity in decision-making, and prioritize product and service improvements.

In a production setting, such a pipeline can serve as a decision-support tool across multiple functions, including product development, quality control, and customer experience management.

Ultimately, this approach illustrates the growing role of Generative AI as a bridge between qualitative data and strategic business actions, helping organizations become more responsive, data-driven, and customer-centric.

In [ ]:
# If running in Google Colab, upload the dataset to the data/ folder or mount Google Drive manually.

In [ ]:
# This cell was used during development and is not required for the portfolio version.

In [ ]:
import os

folder = "data/Colab Notebooks"
[f for f in os.listdir(folder) if f.endswith(".html")]